In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import json
import ast
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

from forge_net.data.dataloaders import GetSingleStepDataLoaders
from forge_net.utils.common import actions_from_feature_map
from forge_net.utils.common import MeshContainer, meshcontainer_to_volume
from forge_net.utils.math import *
import meshio


In [2]:
def process_series(args):
    """Process a single series - this will run in parallel
    
    Args:
        args: Tuple of (series_id, group_df, total_points, press_width, mask_points, seed)
    """
    series_id, group_df, total_points, press_width, mask_points, seed = args
    series_coords_t = []
    series_coords_tp1 = []
    series_steps = []
    series_positions = []
    series_rotations = []
    pv_meshes = []
    pv_meshes_tp1 = []
    bary_coords_list = []
    tri_ids_list = []
    
    for i in range(len(group_df) - 1):
        row_t = group_df.iloc[i]
        row_tp1 = group_df.iloc[i + 1]
       

        # Input mesh (coords from frame i)
        num_steps_t = row_t["solver_steps"]
        vertices_t = np.array(ast.literal_eval(row_t["vertices"])).reshape(-1,3)
        triangles_t = np.array(ast.literal_eval(row_t["triangles"])).reshape(-1,4)
        vertex_temps_t = np.array(ast.literal_eval(row_t["input_temperature"])).reshape(-1,1)
        print(np.array(vertices_t).reshape(-1,3).shape, np.array(triangles_t).reshape(-1,4).shape)

        tmp_mesh_t = meshio.Mesh(
            points=vertices_t, 
            cells=[("tetra", triangles_t)]
        )
        pv_mesh_t = pv.from_meshio(tmp_mesh_t)
        # pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_t)
        # pl.screenshot("tmp.png")

        # print(tmp_mesh_t)
        # print(type(pv_mesh_t)) 

        num_steps_tp1 = row_tp1["solver_steps"]
        vertices_tp1 = np.array(ast.literal_eval(row_tp1["vertices"])).reshape(-1,3)
        triangles_tp1 = np.array(ast.literal_eval(row_tp1["triangles"])).reshape(-1,4)
        vertex_temps_tp1 = np.array(ast.literal_eval(row_tp1["input_temperature"])).reshape(-1,1)
        tmp_mesh_tp1 = meshio.Mesh(
            points=vertices_tp1, 
            cells=[("tetra", triangles_tp1)]
        )
        pv_mesh_tp1 = pv.from_meshio(tmp_mesh_tp1)
 
        # p_tp1 = ((row_tp1["x_max_band"] + row_tp1["x_min_band"]) / 2, 0 , 0)
        # r_tp1 = eulerxyz_to_quat((row_tp1["rotation_euler_x"], 0, 0)) #-> quaternion
        # pv_mesh_t.points = transform_points(np.array(pv_mesh_t.points), np.array(r_tp1), np.array(p_tp1))
        # pv_mesh_tp1.points = transform_points(np.array(pv_mesh_tp1.points), np.array(r_tp1), np.array(p_tp1))

        sampled_points_t, point_triangle_ids, bary_coords, sampled_temps_t = tetrahedral_barycentric_sampling(
                                                                                                            pv_mesh_t, 
                                                                                                            total_points, 
                                                                                                            node_features=vertex_temps_t, 
                                                                                                            seed=seed)
        print("sampled barycenters:" , sampled_points_t)
        tri_ids_list.append(point_triangle_ids)
        bary_coords_list.append(bary_coords)

        sampled_points_tp1, sampled_temps_tp1 = update_tetrahedral_barycentric_points(
                                                                                deformed_mesh=pv_mesh_tp1, 
                                                                                tet_ids=point_triangle_ids, 
                                                                                barycentric_coords=bary_coords, 
                                                                                node_features=vertex_temps_tp1)
        
        pl = pv.Plotter()

        pv_mesh_tp1.point_data["Temperature"] = vertex_temps_tp1.flatten()
        pl.add_mesh(
            pv_mesh_tp1, 
            scalars="Temperature",  # Tell PyVista to color by this attribute
            cmap="coolwarm",         # A great colormap for temperature (Blue to Red)
            show_scalar_bar=True     # Displays the color legend
        )
        pl.show_grid()
        pl.screenshot("tmp_mesh.png")

        pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_tp1,style='wireframe')
        point_cloud = pv.PolyData(sampled_points_tp1)
        point_cloud["temps"] = sampled_temps_t
        pl.add_mesh(point_cloud,
                    scalars='temps',
                    point_size=5.0,render_points_as_spheres=True)
        pl.show_grid()
        pl.screenshot("tmp.png")
        # pl.export_html("tmp.html")

        pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_tp1,style='wireframe')
        point_cloud = pv.PolyData(sampled_points_tp1)
        point_cloud["temps"] = sampled_temps_tp1
        pl.add_mesh(point_cloud,
                    scalars='temps',
                    point_size=5.0,render_points_as_spheres=True)
        pl.show_grid()
        pl.screenshot("tmp_tp1.png")
        # pl.export_html("tmp_tp1.html")


        

In [3]:
# series_id, group_df, total_points, press_width, mask_points, seed = args
db_path="/local/scratch/groves/jax-forgeRL/JAX-FORGE/Agility_Forge_data/data/forge_database.db"
lines=1_000
total_points=10_000
mask_points=False
seed=None
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM hits LIMIT {int(lines)};", conn)
conn.close()

# Prepare arguments for each series
press_width = 1.0
series_ids = df['series_id'].unique()
args_list = [
    (series_id, 
        df[df['series_id'] == series_id].reset_index(drop=True), 
        total_points, 
        press_width,
        mask_points,
        seed
    )
    for series_id in series_ids
]

In [4]:
process_series(args_list[0])

(7531, 3) (30790, 4)
sampled barycenters: [[ 7.64362669  1.56693554 -3.05161826]
 [ 8.50478988  3.34928655  2.36822412]
 [ 4.76527081  3.68355112  1.8023925 ]
 ...
 [28.6967673  -5.74654926 -4.127688  ]
 [91.79025738 -1.54293419 -2.09154566]
 [33.40377447  3.62057966  5.14744648]]


2026-07-07 14:28:53.935 (   1.381s) [    7F80332334C0]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=


(7531, 3) (30790, 4)
sampled barycenters: [[ 8.45028779  0.70493119 -2.08098921]
 [ 9.53201425  4.83167097  2.91156346]
 [ 4.9366887   2.99923995  0.64557934]
 ...
 [47.96897289  1.52635801 -7.53841185]
 [43.54192962  1.02216981  1.78217426]
 [30.17206701  3.45056527  6.51521036]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 8.31506199  1.02319432 -3.68057087]
 [ 9.8519519   3.32732752  3.32250784]
 [ 6.07746627  1.86590328  1.3197493 ]
 ...
 [96.85309889  4.60485794  4.70773278]
 [-4.06126555  7.63810014  1.29563726]
 [97.57332631 -5.60809296 -2.23510925]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 7.80322887  1.15776523 -3.31213512]
 [ 9.18046914  4.43483415  2.29666804]
 [ 4.52285848  1.76896206  0.99007306]
 ...
 [29.71723692  7.53553059 -0.5589469 ]
 [60.79814882 -6.90724286  2.75769625]
 [13.89944599  3.38714732 -1.07191627]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 7.85684444e+00  1.03075201e+00 -2.99113748e+00]
 [ 9.08378678e+00  4.31342248e+00  2.50603731e+00]
 [ 4.33619